# Проект спринта 7  
**Цель проекта:**   
- Провести исследовательский анализ данных о видеоиграх

 
**Задачи:**
- Привести названия столбцов к единому стилю.
- Обработать пропущенные значения и дубликаты.
- Привести данные к корректным типам.
- Сформировать срез данных по релевантному периоду (2000–2013).
- Выделить топ-7 платформ по количеству игр, выпущенных за весь требуемый период.


### Описание данных
/datasets/new_games.csv содержат информацию о продажах игр разных жанров и платформ, а также пользовательские и экспертные оценки игр:  
Name — название игры.  
Platform — название платформы.  
Year of Release — год выпуска игры.  
Genre — жанр игры.  
NA sales — продажи в Северной Америке (в миллионах проданных копий).  
EU sales — продажи в Европе (в миллионах проданных копий).  
JP sales — продажи в Японии (в миллионах проданных копий).  
Other sales — продажи в других странах (в миллионах проданных копий).  
Critic Score — оценка критиков (от 0 до 100).  
User Score — оценка пользователей (от 0 до 10).  
Rating — рейтинг организации ESRB (англ. Entertainment Software Rating Board). Эта ассоциация определяет рейтинг компьютерных игр и присваивает им подходящую возрастную категорию.

In [40]:
import pandas as pd

In [41]:
# Создание датасета

df = pd.read_csv('https://code.s3.yandex.net/datasets/new_games.csv')
col_num = df.shape[0]
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   Name             16954 non-null  object 
 1   Platform         16956 non-null  object 
 2   Year of Release  16681 non-null  float64
 3   Genre            16954 non-null  object 
 4   NA sales         16956 non-null  float64
 5   EU sales         16956 non-null  object 
 6   JP sales         16956 non-null  object 
 7   Other sales      16956 non-null  float64
 8   Critic Score     8242 non-null   float64
 9   User Score       10152 non-null  object 
 10  Rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB


Стоит заметить что во многих строках содержатся пропуски и для некоторых строк тип данных не соответсвует содержанию
Для корректной работы с year_of_release стоит привести поле к int
Также странно что поле user_score имеет тип object а не float, аналогично поля eu_sales и jp_sales

### Проверка ошибок в данных и их предобработка

In [42]:
# Приведем названия столбцов к формату snake_case

df.columns = (df.columns.str.strip().str.lower().str.replace(r"\s+", "_", regex=True))
df.info()

# Изменим типы некоторых полей на более подходящие

df['year_of_release'] = pd.to_numeric(df['year_of_release'], errors='coerce').astype('Int64')
df['user_score'] = pd.to_numeric(df['user_score'], errors='coerce')
df['eu_sales'] = pd.to_numeric(df['eu_sales'], errors='coerce')
df['jp_sales'] = pd.to_numeric(df['jp_sales'], errors='coerce')
df['genre'] = df['genre'].astype('category')
df['platform'] = df['platform'].astype('category')
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype  
---  ------           --------------  -----  
 0   name             16954 non-null  object 
 1   platform         16956 non-null  object 
 2   year_of_release  16681 non-null  float64
 3   genre            16954 non-null  object 
 4   na_sales         16956 non-null  float64
 5   eu_sales         16956 non-null  object 
 6   jp_sales         16956 non-null  object 
 7   other_sales      16956 non-null  float64
 8   critic_score     8242 non-null   float64
 9   user_score       10152 non-null  object 
 10  rating           10085 non-null  object 
dtypes: float64(4), object(7)
memory usage: 1.4+ MB
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 16956 entries, 0 to 16955
Data columns (total 11 columns):
 #   Column           Non-Null Count  Dtype   
---  ------           --------------  -----   
 0   name             16954 non-null  

In [43]:
# Поиск абсолютного и относительного кол-ва пропусков

print('Абсолютное число пропусков:')
print(df.isna().sum())
print('\nОтносительное число пропусков:')
print(df.isna().sum() / df.shape[0])

Абсолютное число пропусков:
name                  2
platform              0
year_of_release     275
genre                 2
na_sales              0
eu_sales              6
jp_sales              4
other_sales           0
critic_score       8714
user_score         9268
rating             6871
dtype: int64

Относительное число пропусков:
name               0.000118
platform           0.000000
year_of_release    0.016218
genre              0.000118
na_sales           0.000000
eu_sales           0.000354
jp_sales           0.000236
other_sales        0.000000
critic_score       0.513918
user_score         0.546591
rating             0.405225
dtype: float64


Пропуски содержаться во многих полях, но стоит заметить что в полях name, year_of_release, eu_sales и jp_sales число пропусков незначительно проще всего их удалить, не опасаясь влияния на результат. Строки поля name содержащие пропуски можно удалить т.к восстановить их нет воможности. В поле year_of_release пропуски составляют более 1%, в данном случае можно как удалить данные, так и оставить их в текущем виде, по согласованию с заказчиком.
Поля: critic_score, user_score и rating содержат наибольшее число пропусков. Удялять данные строки строго не рекомендуется, можно заменить пропущенные значения средним по аналогичным параметрам.

| Поле     | Пропуски | Предполагаемая причина     |
|---------------|---------------|---------------|
| name  | 2 | Ошибки при сборе или экспорте данных |
| year_of_release | 275 | Неизвестная дата выпуска |
| genre | 2 | Игра с неопределённым жанром  |
| eu_sales | 6 | Игра не вышла в этом регионе или отсутствуют данные по продажам. |
| jp_sales | 4 | Игра не вышла в этом регионе или отсутствуют данные по продажам. |
| critic_score | 8714 | Игра не получила ни одной рецензии или отзывы могли быть удалены с источников |
| user_score | 9268 | Игра не получила ни одной рецензии или отзывы могли быть удалены с источников   |
| rating | 6871 | Игра не проходила рейтинговую сертификацию |

In [44]:
# Удаление пропусков

df = df.dropna(subset=['name', 'genre', 'eu_sales', 'jp_sales'])
df.isna().sum()

name                  0
platform              0
year_of_release     275
genre                 0
na_sales              0
eu_sales              0
jp_sales              0
other_sales           0
critic_score       8710
user_score         9264
rating             6868
dtype: int64

In [45]:
# Замена пропусков на индикаторное значение

df['critic_score'] = df['critic_score'].fillna(-1)
df['user_score'] = df['user_score'].fillna(-1)
df['rating'] = df['rating'].fillna('')
df.isna().sum()


name                 0
platform             0
year_of_release    275
genre                0
na_sales             0
eu_sales             0
jp_sales             0
other_sales          0
critic_score         0
user_score           0
rating               0
dtype: int64

Пропуски в полях critic_score и critic_score я заменил на значение-индикатор.  
Поле year_of_release я оставил без изменений.
Пропуски в поле rating были заменены на ''. 

### Явные и неявные дубликаты в данных

In [46]:
for col in ['name','platform', 'genre','year_of_release', 'rating']:
    print(f"{col}: {df[col].nunique()} уникальных значений")

name: 11556 уникальных значений
platform: 31 уникальных значений
genre: 24 уникальных значений
year_of_release: 37 уникальных значений
rating: 9 уникальных значений


In [47]:
# Выведем все уникальные значения
for col in ['platform', 'genre','year_of_release', 'rating']:
    print(f"{col}: {df[col].unique().tolist()}")

platform: ['Wii', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA', 'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XOne', 'WiiU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16', '3DO', 'GG', 'PCFX']
genre: ['Sports', 'Platform', 'Racing', 'Role-Playing', 'Puzzle', 'Misc', 'Shooter', 'Simulation', 'Action', 'Fighting', 'Adventure', 'Strategy', 'MISC', 'ROLE-PLAYING', 'RACING', 'ACTION', 'SHOOTER', 'FIGHTING', 'SPORTS', 'PLATFORM', 'ADVENTURE', 'SIMULATION', 'PUZZLE', 'STRATEGY']
year_of_release: [2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010, 2013, 2004, 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014, 1992, 1997, 1993, 1994, 1982, 2016, 2003, 1986, 2000, <NA>, 1995, 1991, 1981, 1987, 1980, 1983]
rating: ['E', '', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']


platform: не содержит неявных дубликатов  
genre: содержит неявные дублкаты связанные с разным способом написания  
year_of_release: не содержит дубликатов  
rating: не содержит дубликатов

In [48]:
# Уберем неявные дубликаты
df['genre'] = df['genre'].str.lower()
df['platform'] = df['platform'].str.upper()

# Выведем все уникальные значения
for col in ['platform', 'genre','year_of_release', 'rating']:
    print(f"{col}: {df[col].unique().tolist()}")

# Проверим наличие явных дубликатов
duplicates = [i for i in df.duplicated(keep='first') if i]
print(f"Число явных дубликатов: {len(duplicates)}")

# Удалим явные дубликаты
df.drop_duplicates(subset=None, keep='first', inplace=True, ignore_index=False)


platform: ['WII', 'NES', 'GB', 'DS', 'X360', 'PS3', 'PS2', 'SNES', 'GBA', 'PS4', '3DS', 'N64', 'PS', 'XB', 'PC', '2600', 'PSP', 'XONE', 'WIIU', 'GC', 'GEN', 'DC', 'PSV', 'SAT', 'SCD', 'WS', 'NG', 'TG16', '3DO', 'GG', 'PCFX']
genre: ['sports', 'platform', 'racing', 'role-playing', 'puzzle', 'misc', 'shooter', 'simulation', 'action', 'fighting', 'adventure', 'strategy']
year_of_release: [2006, 1985, 2008, 2009, 1996, 1989, 1984, 2005, 1999, 2007, 2010, 2013, 2004, 1990, 1988, 2002, 2001, 2011, 1998, 2015, 2012, 2014, 1992, 1997, 1993, 1994, 1982, 2016, 2003, 1986, 2000, <NA>, 1995, 1991, 1981, 1987, 1980, 1983]
rating: ['E', '', 'M', 'T', 'E10+', 'K-A', 'AO', 'EC', 'RP']
Число явных дубликатов: 241


In [49]:
print(f"Абсолютное число удаленных строк: {col_num-df.shape[0]}")
print(f"Относительное число удаленных строк: {(col_num-df.shape[0]) / df.shape[0]}")

Абсолютное число удаленных строк: 253
Относительное число удаленных строк: 0.015146979584505777


 ### Общий промежуточный вывод

На этом этапе была выполнена предварительная обработка данных:  
Приведены к единому стилю названия столбцов (в нижний регистр, с использованием snake_case)
Удалены явные и простые неявные дубликаты  
Обнаружены и обработаны пропущенные значения:
В столбце critic_score и user_score пропуски были заменены на средние значения по сочетанию platform и genre  
В столбце rating пропуски были заменены на пустую строку  
Типы данных приведены к корректным

### Фильтрация данных

In [50]:
df = df[df['year_of_release'].between(2000, 2013, inclusive='both')]
df.shape[0]

12772

### Категоризация данных
Категоризовать игры по оценкам пользователей и экспертов. Выделите три категории

In [51]:
def categorize_user_rating(row):
    if (row['user_score'] >= 8): 
        return 'высокая оценка'
    elif (row['user_score'] >= 3):
        return 'средняя оценка'
    else:
        return 'низкая оценка'

def categorize_critic_rating(row):
    if (row['critic_score'] >= 80): 
        return 'высокая оценка'
    elif (row['critic_score'] >= 30):
        return 'средняя оценка'
    else:
        return 'низкая оценка'
        
df['user_rating'] = df.apply(categorize_user_rating, axis=1)
df['critic_rating'] = df.apply(categorize_critic_rating, axis=1)

In [52]:
print(df)

                                                   name platform  \
0                                            Wii Sports      WII   
2                                        Mario Kart Wii      WII   
3                                     Wii Sports Resort      WII   
6                                 New Super Mario Bros.       DS   
7                                              Wii Play      WII   
...                                                 ...      ...   
16947                     Men in Black II: Alien Escape       GC   
16949                Woody Woodpecker in Crazy Castle 5      GBA   
16950  SCORE International Baja 1000: The Official Game      PS2   
16952                                  LMA Manager 2007     X360   
16954                                  Spirits & Spells      GBA   

       year_of_release     genre  na_sales  eu_sales  jp_sales  other_sales  \
0                 2006    sports     41.36     28.96      3.77         8.45   
2                 2008   

In [53]:
top_platforms = (df['platform'].value_counts().head(7))

print(top_platforms)

platform
PS2     2126
DS      2117
WII     1275
PSP     1179
X360    1118
PS3     1087
GBA      810
Name: count, dtype: int64


Был сформирован срез, охватывающий игры, выпущенные в период с 2000 по 2013 год включительно. Это обусловлено тем, что до 2000 года и после 2013 года данных значительно меньше

## Общий вывод

Таким образом была проведена работа по подготовке и обзору данных. Были проведены:  
Проверка ошибок в данных и их предобработка - были изучены названия столбцов и типы данных, проведена проверка наличия пропусков в данных, а также наличие явных и неявных дубликаты в данных

Фильтрация данных - реализована категоризация данных на игры с "высокой", "средней" и "низкой" оценкой от пользователей и критиков. Были добавлены поля "user_rating" и "critic_rating". А также выделены топ-7 платформ по количеству игр, выпущенных за весь требуемый период.